In [1]:
!pip install -q transformers datasets evaluate scikit-learn pandas tqdm gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00


In [2]:
import torch, sys
print("Torch:", getattr(torch, "__version__", "not installed"))
print("CUDA available:", torch.cuda.is_available())
# Optional: show GPU details (may fail on some runtimes)
!nvidia-smi || echo "nvidia-smi not available"


Torch: 2.8.0+cu126
CUDA available: True
Sat Sep 27 07:07:17 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
from transformers import pipeline

# Sentiment model (SST-2, binary sentiment)
sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Emotion model (GoEmotions, 27 labels)
emotion_pipe = pipeline(
    "text-classification",
    model="bhadresh-savani/distilbert-base-uncased-emotion",
    return_all_scores=True
)

# Test
print(sentiment_pipe("I love this project!"))
print(emotion_pipe("I am frustrated and also a little excited"))


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cuda:0
Device set to use cuda:0


[{'label': 'POSITIVE', 'score': 0.9998873472213745}]
[[{'label': 'sadness', 'score': 0.0004322142631281167}, {'label': 'joy', 'score': 0.00040026975329965353}, {'label': 'love', 'score': 0.00018326137796975672}, {'label': 'anger', 'score': 0.9974094033241272}, {'label': 'fear', 'score': 0.0013944549718871713}, {'label': 'surprise', 'score': 0.0001803324557840824}]]


In [6]:
# Cell 5 — load datasets (small samples recommended)
from datasets import load_dataset
imdb = load_dataset("imdb")
go = load_dataset("go_emotions")  # GoEmotions

# Inspect a sample to see fields & label format
print(imdb['train'][0])
print(go['train'][0])   # check the 'labels' field and structure


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

...

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [9]:
# cell: safe batched_pipeline (replace your old one)
from tqdm import tqdm
import numpy as np

def batched_pipeline(pipeline_fn, texts, batch_size=32, max_length=512, truncation=True):
    """
    Calls a HF pipeline on texts in batches, requesting tokenizer truncation.
    If a pipeline call fails, falls back to per-example truncation using the pipeline's tokenizer.
    """
    out = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        try:
            # Forward tokenizer args to the pipeline (this usually works)
            out_batch = pipeline_fn(batch, truncation=truncation, max_length=max_length)
        except Exception as e:
            # Fallback: explicit tokenization + decode -> safe, short strings
            tok = getattr(pipeline_fn, "tokenizer", None)
            if tok is None:
                raise  # nothing we can do
            safe_batch = []
            for t in batch:
                ids = tok.encode(t, truncation=True, max_length=max_length)
                safe_text = tok.decode(ids, skip_special_tokens=True)
                safe_batch.append(safe_text)
            out_batch = pipeline_fn(safe_batch)
        out.extend(out_batch)
    return out


In [10]:
# cell: sentiment evaluation (updated)
from sklearn.metrics import accuracy_score, classification_report
import random

# small sample to keep it fast
n = 500
test_sample = imdb['test'].shuffle(seed=42).select(range(n))
texts = [ex['text'] for ex in test_sample]
y_true = [ex['label'] for ex in test_sample]  # 0 = neg, 1 = pos

# use a shorter max_length to avoid truncation warnings and speed things up
sent_preds = batched_pipeline(sentiment_pipe, texts, batch_size=32, max_length=256)

# map pipeline labels to 0/1 (depends on the sentiment model; adjust if different)
label_map = {"NEGATIVE":0, "POSITIVE":1}
y_pred = [label_map[p['label'].upper()] for p in sent_preds]

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=["NEG","POS"]))


100%|██████████| 16/16 [00:04<00:00,  3.89it/s]

Accuracy: 0.876
              precision    recall  f1-score   support

         NEG       0.86      0.90      0.88       254
         POS       0.89      0.85      0.87       246

    accuracy                           0.88       500
   macro avg       0.88      0.88      0.88       500
weighted avg       0.88      0.88      0.88       500



In [12]:
# Cell 8 — evaluate emotions on a small sample (GoEmotions is multi-label)
from sklearn.metrics import f1_score, accuracy_score, classification_report
import numpy as np

# sample size (small to keep runtime fast)
n = 400
sample = go['test'].shuffle(seed=42).select(range(n))

texts = [ex['text'] for ex in sample]
# true labels are lists of label indices
dataset_label_names = go['train'].features['labels'].feature.names
num_labels = len(dataset_label_names)
print("Number of emotion labels:", num_labels)

# get model outputs (batched)
emotion_raw = batched_pipeline(emotion_pipe, texts, batch_size=16)

# check order of labels from model
model_label_order = [d['label'] for d in emotion_raw[0]]
print("Model label order (first 10):", model_label_order[:10])

# Create mapping from model labels -> dataset index
label2idx = {lab: i for i, lab in enumerate(dataset_label_names)}

# Build y_true (binary matrix)
y_true = np.zeros((n, num_labels), dtype=int)
for i, ex in enumerate(sample):
    for lab in ex['labels']:
        y_true[i, lab] = 1

# Build score matrix aligned to dataset label order
scores = np.zeros((n, num_labels))
for i, row in enumerate(emotion_raw):
    for d in row:
        if d['label'] in label2idx:  # only use known labels
            scores[i, label2idx[d['label']]] = d['score']

# choose threshold (tuneable)
threshold = 0.30
y_pred = (scores >= threshold).astype(int)

# metrics
micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
subset_acc = accuracy_score(y_true, y_pred)
print(f"Micro-F1 (threshold {threshold}):", micro_f1)
print("Subset accuracy (exact match):", subset_acc)

# Per-label report (optional)
print(classification_report(y_true, y_pred, target_names=dataset_label_names, zero_division=0))


Number of emotion labels: 28


100%|██████████| 25/25 [00:02<00:00, 11.35it/s]

Model label order (first 10): ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
Micro-F1 (threshold 0.3): 0.07982261640798226
Subset accuracy (exact match): 0.0575
                precision    recall  f1-score   support

    admiration       0.00      0.00      0.00        38
     amusement       0.00      0.00      0.00        18
         anger       0.03      0.80      0.05         5
     annoyance       0.00      0.00      0.00        26
      approval       0.00      0.00      0.00        27
        caring       0.00      0.00      0.00         9
     confusion       0.00      0.00      0.00        10
     curiosity       0.00      0.00      0.00        24
        desire       0.00      0.00      0.00         3
disappointment       0.00      0.00      0.00        14
   disapproval       0.00      0.00      0.00        18
       disgust       0.00      0.00      0.00         6
 embarrassment       0.00      0.00      0.00         4
    excitement       0.00      0.00      0.00

In [14]:
# Cell 9 — save results
import pandas as pd

# ---- Sentiment (IMDB) ----
# Ensure we define test_sample from IMDB test split again (safe even if already defined)
n = 400
test_sample = imdb['test'].shuffle(seed=42).select(range(n))

# get true labels and texts
texts_imdb = [ex['text'] for ex in test_sample]
true_sent = [ex['label'] for ex in test_sample]

# predicted sentiment (reuse from earlier y_pred_sent if you saved it)
# If not saved, just recompute quickly
sent_preds = batched_pipeline(sentiment_pipe, texts_imdb, batch_size=32)
y_pred_sent = [1 if p['label'] == 'POSITIVE' else 0 for p in sent_preds]

df_imdb = pd.DataFrame({
    "text": texts_imdb,
    "sent_true": true_sent,
    "sent_pred": y_pred_sent
})
df_imdb.to_csv("/content/imdb_sample_preds.csv", index=False)
print("✅ Saved /content/imdb_sample_preds.csv")

# ---- Emotions (GoEmotions) ----
emotion_df = pd.DataFrame({
    "text": texts,   # from Step 8
    "true_labels": [list(np.where(row == 1)[0]) for row in y_true],
    "pred_labels": [list(np.where(row == 1)[0]) for row in y_pred],
})
emotion_df.to_csv("/content/goemotions_sample_preds.csv", index=False)
print("✅ Saved /content/goemotions_sample_preds.csv")


100%|██████████| 13/13 [00:04<00:00,  3.24it/s]

✅ Saved /content/imdb_sample_preds.csv
✅ Saved /content/goemotions_sample_preds.csv


In [15]:
# Cell 10 — a tiny Gradio UI (fast demo)
import gradio as gr

def analyze(text):
    s = sentiment_pipe(text)[0]
    e = emotion_pipe(text)[0]   # list of dicts
    # get top 3 emotions
    top3 = sorted(e, key=lambda x: x['score'], reverse=True)[:3]
    top3 = [(d['label'], float(d['score'])) for d in top3]
    return f"{s['label']} ({s['score']:.2f})", top3

iface = gr.Interface(fn=analyze,
                     inputs=gr.Textbox(lines=4, placeholder="Type some text..."),
                     outputs=[gr.Textbox(label="Sentiment"), gr.JSON(label="Top emotions (label,score)")],
                     title="Sentiment & Emotion Demo")
# launch with share=True to get an external URL (Gradio provides it)
iface.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://79e9415b71029b7fbb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
